# UTS Prediksi Modern & Machine Learning
## Experiment Challenge: Credit Default Prediction



## 0. Setup library

In [11]:
# ==========================================
# LOAD DATASET FROM GOOGLE DRIVE
# ==========================================

from google.colab import drive
import pandas as pd
import os

# Connect Google Drive
drive.mount('/content/drive')

# Lokasi dataset di Google Drive
FILE_PATH = "/content/drive/MyDrive/UCI_Credit_Card.csv"

# Cek apakah file tersedia
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di: {FILE_PATH}\n"
        "Pastikan file UCI_Credit_Card.csv sudah berada di MyDrive."
    )

# Load dataset
df = pd.read_csv(FILE_PATH)

print("Dataset berhasil dimuat!")
print("File:", FILE_PATH)
print("Dimensi dataset:", df.shape)

display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset berhasil dimuat!
File: /content/drive/MyDrive/UCI_Credit_Card.csv
Dimensi dataset: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [12]:
import warnings, os, glob, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Setup selesai.")

Setup selesai.


## Experiment 0 — Data Understanding & Preparation

### 0.1 Membaca dataset

In [14]:
# Load dataset
FILE_PATH = "UCI_Credit_Card.csv"

df = pd.read_csv(FILE_PATH)

print("File:", FILE_PATH)
print("Dimensi awal:", df.shape)
display(df.head())

File: UCI_Credit_Card.csv
Dimensi awal: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [15]:
print("Informasi struktur:")
display(pd.DataFrame({
    "tipe_data": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "jumlah_unik": df.nunique()
}))
print("\nJumlah baris duplikat:", df.duplicated().sum())
print("\nNama kolom:")
print(df.columns.tolist())

Informasi struktur:


,tipe_data,missing,jumlah_unik
ID,int64,0,30000
LIMIT_BAL,float64,0,81
SEX,int64,0,2
EDUCATION,int64,0,7
MARRIAGE,int64,0,4
AGE,int64,0,56
PAY_0,int64,0,11
PAY_2,int64,0,11
PAY_3,int64,0,11
PAY_4,int64,0,11



Jumlah baris duplikat: 0

Nama kolom:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default.payment.next.month']


### 0.2 Menentukan target, mengecek kualitas data, dan distribusi target

In [16]:
# Bersihkan spasi pada nama kolom (jika ada)
df.columns = df.columns.astype(str).str.strip()

TARGET = "default.payment.next.month"
if TARGET not in df.columns:
    # Beberapa salinan dataset mungkin menggunakan variasi nama target.
    alternatives = [c for c in df.columns if "default" in c.lower() and ("next" in c.lower() or "payment" in c.lower())]
    if len(alternatives) == 1:
        TARGET = alternatives[0]
    else:
        raise ValueError(f"Kolom target '{TARGET}' tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")

print("Target yang digunakan:", TARGET)
print("\nDistribusi target:")
display(df[TARGET].value_counts(dropna=False).rename_axis("kelas").to_frame("jumlah"))
display((df[TARGET].value_counts(normalize=True, dropna=False)*100).round(2).rename("persentase (%)").to_frame())

print("\nNilai target unik:", sorted(df[TARGET].dropna().unique().tolist()))
print("\nStatistik numerik:")
display(df.describe().T)

Target yang digunakan: default.payment.next.month

Distribusi target:


,jumlah
kelas,
0,23364
1,6636


,persentase (%)
default.payment.next.month,
0,77.88
1,22.12



Nilai target unik: [0, 1]

Statistik numerik:


,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


### 0.3 Pemeriksaan dan keputusan persiapan data

- Kolom `ID` tidak digunakan sebagai predictor.
- Baris duplikat dan missing value diperiksa terlebih dahulu; jangan otomatis menghapus baris tanpa alasan.
- Target harus berupa kelas 0 dan 1.
- Split dilakukan sebelum fitting preprocessing/model untuk menghindari data leakage.
- Preprocessing dimasukkan ke dalam `Pipeline`, sehingga imputer/scaler hanya dipelajari dari data training pada setiap fold.

In [17]:
# Buat salinan data; hapus duplikat hanya jika memang ditemukan dan diputuskan relevan.
print("Duplikat sebelum keputusan:", int(df.duplicated().sum()))
print("Missing value total:", int(df.isna().sum().sum()))

# Buang baris yang targetnya kosong karena target tidak dapat dipakai untuk supervised learning.
df = df.dropna(subset=[TARGET]).copy()

# Pastikan target biner sesuai ketentuan tugas.
target_values = set(pd.Series(df[TARGET]).dropna().unique())
if not target_values.issubset({0, 1, 0.0, 1.0}):
    raise ValueError(f"Target bukan biner 0/1. Nilai ditemukan: {target_values}")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# Buang kolom ID dari predictor (case-insensitive).
id_cols = [c for c in X.columns if c.strip().lower() in ["id", "unnamed: 0"]]
X = X.drop(columns=id_cols, errors="ignore")

# Kenali kolom kategorikal/numerik berdasarkan tipe data.
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

print("Dimensi X:", X.shape)
print("Distribusi target:")
display(y.value_counts().sort_index().rename("jumlah").to_frame())
print("Kolom ID yang dikeluarkan:", id_cols)
print("Jumlah fitur numerik:", len(numeric_features))
print("Jumlah fitur kategorikal:", len(categorical_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Duplikat sebelum keputusan: 0
Missing value total: 0
Dimensi X: (30000, 23)
Distribusi target:


,jumlah
default.payment.next.month,
0,23364
1,6636


Kolom ID yang dikeluarkan: ['ID']
Jumlah fitur numerik: 23
Jumlah fitur kategorikal: 0
Train: (24000, 23) | Test: (6000, 23)


## Fungsi evaluasi yang dipakai untuk semua eksperimen

In [18]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(steps=numeric_steps)

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_pipe, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_pipe, categorical_features))
    return ColumnTransformer(transformers=transformers, remainder="drop")

def evaluate_model(name, model, preprocessor=None):
    steps = []
    if preprocessor is not None:
        steps.append(("preprocess", preprocessor))
    steps.append(("model", model))
    pipe = Pipeline(steps=steps)

    start = time.time()
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=CV, scoring="f1", n_jobs=-1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    elapsed = time.time() - start

    result = {
        "Experiment": name,
        "CV F1-score": cv_scores.mean(),
        "CV F1 std": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test Precision": precision_score(y_test, pred, zero_division=0),
        "Test Recall": recall_score(y_test, pred, zero_division=0),
        "Test F1-score": f1_score(y_test, pred, zero_division=0),
        "Waktu (detik)": elapsed
    }
    print(f"\n{name}")
    print("CV F1 per fold:", np.round(cv_scores, 4))
    print(classification_report(y_test, pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    return pipe, result

## Experiment 1 — Baseline Logistic Regression
Baseline menggunakan Logistic Regression tanpa scaling fitur. Imputasi tetap dimasukkan ke pipeline agar missing value tidak menyebabkan error.

In [19]:
baseline_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
baseline_pipe, result_1 = evaluate_model(
    "Experiment 1 - Baseline Logistic Regression",
    baseline_model,
    preprocessor=make_preprocessor(scale_numeric=False)
)


Experiment 1 - Baseline Logistic Regression
CV F1 per fold: [0.3538 0.2818 0.3731 0.3404 0.3322]
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      4673
           1       0.68      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.75      0.61      0.62      6000
weighted avg       0.79      0.81      0.77      6000

Confusion matrix:
 [[4523  150]
 [1004  323]]


## Experiment 2 — Preprocessing
Di eksperimen ini, Logistic Regression menggunakan scaling fitur numerik. Bandingkan F1-score dengan Experiment 1.

In [20]:
scaled_lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
scaled_pipe, result_2 = evaluate_model(
    "Experiment 2 - Logistic Regression + Scaling",
    scaled_lr,
    preprocessor=make_preprocessor(scale_numeric=True)
)


Experiment 2 - Logistic Regression + Scaling
CV F1 per fold: [0.3661 0.3732 0.3807 0.364  0.3395]
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      4673
           1       0.69      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.75      0.60      0.62      6000
weighted avg       0.79      0.81      0.77      6000

Confusion matrix:
 [[4528  145]
 [1009  318]]


## Experiment 3 — Model Comparison
Bandingkan Logistic Regression, Decision Tree, dan Random Forest menggunakan prosedur evaluasi yang sama.

In [21]:
result_3_rows = []
model_pipes = {}

models_exp3 = [
    ("Logistic Regression", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), make_preprocessor(scale_numeric=True)),
    ("Decision Tree", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"), make_preprocessor(scale_numeric=False)),
    ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"), make_preprocessor(scale_numeric=False))
]

for name, model, prep in models_exp3:
    pipe, result = evaluate_model(f"Experiment 3 - {name}", model, prep)
    model_pipes[name] = pipe
    result["Model"] = name
    result_3_rows.append(result)

results_exp3 = pd.DataFrame(result_3_rows).sort_values("CV F1-score", ascending=False)
display(results_exp3[["Model", "CV F1-score", "Test Accuracy", "Test Precision", "Test Recall", "Test F1-score", "Waktu (detik)"]])
best_model_name = results_exp3.iloc[0]["Model"]
print("Model terbaik berdasarkan CV F1-score:", best_model_name)


Experiment 3 - Logistic Regression
CV F1 per fold: [0.3661 0.3732 0.3807 0.364  0.3395]
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      4673
           1       0.69      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.75      0.60      0.62      6000
weighted avg       0.79      0.81      0.77      6000

Confusion matrix:
 [[4528  145]
 [1009  318]]

Experiment 3 - Decision Tree
CV F1 per fold: [0.3635 0.3983 0.3842 0.41   0.3798]
              precision    recall  f1-score   support

           0       0.83      0.83      0.83      4673
           1       0.39      0.38      0.39      1327

    accuracy                           0.73      6000
   macro avg       0.61      0.61      0.61      6000
weighted avg       0.73      0.73      0.73      6000

Confusion matrix:
 [[3875  798]
 [ 819  508]]

Experiment 3 - Random Forest
CV F1 per fold: [0.4571 0.4634 0.4564 0.4574 0.4557]

,Model,CV F1-score,Test Accuracy,Test Precision,Test Recall,Test F1-score,Waktu (detik)
2,Random Forest,0.458024,0.812000,0.640339,0.342125,0.445972,88.707278
1,Decision Tree,0.387154,0.730500,0.388974,0.382818,0.385872,3.608862
0,Logistic Regression,0.364699,0.807667,0.686825,0.239638,0.355307,1.010234


Model terbaik berdasarkan CV F1-score: Random Forest


## Experiment 4 — Hyperparameter Tuning
GridSearchCV menggunakan 5-fold CV dan F1-score. Grid sengaja dibatasi agar waktu komputasi tetap masuk akal.

In [22]:
if best_model_name == "Logistic Regression":
    tuning_pipeline = Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=True)),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
    param_grid = {
        "model__C": [0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"]
    }
elif best_model_name == "Decision Tree":
    tuning_pipeline = Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])
    param_grid = {
        "model__max_depth": [4, 8, 12, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__class_weight": [None, "balanced"]
    }
else:
    tuning_pipeline = Pipeline([
        ("preprocess", make_preprocessor(scale_numeric=False)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
    ])
    param_grid = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 12],
        "model__min_samples_leaf": [1, 5],
        "model__class_weight": [None, "balanced_subsample"]
    }

grid = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=CV,
    n_jobs=-1,
    refit=True,
    verbose=1
)
start = time.time()
grid.fit(X_train, y_train)
tuning_time = time.time() - start
tuned_pipe = grid.best_estimator_
tuned_pred = tuned_pipe.predict(X_test)

result_4 = {
    "Experiment": "Experiment 4 - Tuned " + best_model_name,
    "CV F1-score": grid.best_score_,
    "CV F1 std": np.nan,
    "Test Accuracy": accuracy_score(y_test, tuned_pred),
    "Test Precision": precision_score(y_test, tuned_pred, zero_division=0),
    "Test Recall": recall_score(y_test, tuned_pred, zero_division=0),
    "Test F1-score": f1_score(y_test, tuned_pred, zero_division=0),
    "Waktu (detik)": tuning_time
}
print("Best parameters:", grid.best_params_)
print("Best CV F1-score:", round(grid.best_score_, 4))
print(classification_report(y_test, tuned_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, tuned_pred))
display(pd.DataFrame([result_4]))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters: {'model__class_weight': 'balanced_subsample', 'model__max_depth': 12, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}
Best CV F1-score: 0.5454
              precision    recall  f1-score   support

           0       0.87      0.86      0.86      4673
           1       0.53      0.56      0.54      1327

    accuracy                           0.79      6000
   macro avg       0.70      0.71      0.70      6000
weighted avg       0.80      0.79      0.79      6000

Confusion matrix:
 [[4003  670]
 [ 583  744]]


,Experiment,CV F1-score,CV F1 std,Test Accuracy,Test Precision,Test Recall,Test F1-score,Waktu (detik)
0,Experiment 4 - Tuned Random Forest,0.545406,NaN,0.791167,0.526167,0.560663,0.542868,704.897621


## Experiment 5 — Advanced Model
Model advanced yang digunakan adalah Gradient Boosting. Jika runtime terlalu lama, tunggu proses selesai; jangan mengubah test set untuk memilih model.

In [23]:
advanced_model = GradientBoostingClassifier(random_state=RANDOM_STATE)
advanced_pipe, result_5 = evaluate_model(
    "Experiment 5 - Gradient Boosting",
    advanced_model,
    preprocessor=make_preprocessor(scale_numeric=False)
)


Experiment 5 - Gradient Boosting
CV F1 per fold: [0.4747 0.4895 0.4739 0.4849 0.47  ]
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.66      0.36      0.47      1327

    accuracy                           0.82      6000
   macro avg       0.75      0.66      0.68      6000
weighted avg       0.80      0.82      0.80      6000

Confusion matrix:
 [[4429  244]
 [ 846  481]]


## Final Analysis — Ringkasan hasil eksperimen

In [24]:
# Susun ringkasan utama. Experiment 3 berisi tiga model, sesuai permintaan perbandingan.
summary_rows = [result_1, result_2]
summary_rows.extend(result_3_rows)
summary_rows.extend([result_4, result_5])
summary = pd.DataFrame(summary_rows)

display(summary[[
    "Experiment", "CV F1-score", "Test Accuracy", "Test Precision", "Test Recall", "Test F1-score"
]].round(4))

# Ekspor tabel agar mudah dipakai dalam laporan PDF.
summary.to_csv("summary_experiments.csv", index=False)
print("Tabel disimpan sebagai summary_experiments.csv")

,Experiment,CV F1-score,Test Accuracy,Test Precision,Test Recall,Test F1-score
0,Experiment 1 - Baseline Logistic Regression,0.3363,0.8077,0.6829,0.2434,0.3589
1,Experiment 2 - Logistic Regression + Scaling,0.3647,0.8077,0.6868,0.2396,0.3553
2,Experiment 3 - Logistic Regression,0.3647,0.8077,0.6868,0.2396,0.3553
3,Experiment 3 - Decision Tree,0.3872,0.7305,0.3890,0.3828,0.3859
4,Experiment 3 - Random Forest,0.4580,0.8120,0.6403,0.3421,0.4460
5,Experiment 4 - Tuned Random Forest,0.5454,0.7912,0.5262,0.5607,0.5429
6,Experiment 5 - Gradient Boosting,0.4786,0.8183,0.6634,0.3625,0.4688


Tabel disimpan sebagai summary_experiments.csv
